# `microbetag`: metabolic secrets behind microbial co-occurrence

## How to use this notebook

> ⚠️ Attention! 
>
> Before anything else, run the following chunk every time you reload, exit or change kernel (see below).

In [2]:
# Load basic Python libraries
import os 
import sys
from pathlib import Path
notebook_dir = Path(os.getcwd())

Now, let us introduce ourselves! 🤭

This notebook introduces a set of concepts and tasks using [`microbetag`](https://microbetag.readthedocs.io/) — a microbial co-occurrence network annotator.  
By combining `microbetag` with network clustering algorithms and enrichment analysis, one can enhance the interpretation of microbial networks and generate new hypotheses about the processes driving the communities under study.

`microbetag` can be used in several ways, but most importantly, it should be seen as a **pipeline**.  
It needs to:

- infer a network from an abundance table (if a network is not already provided),  
- map taxonomies to the closest genomes (if custom genomes are not provided),  
- predict phenotypic traits for taxa based on the available genomes,  
- reconstruct Genome-Scale Metabolic Models (GEMs) from genomes (if not already provided) to infer:  
  - metabolic compounds that must be acquired exogenously (*seeds*), and  
  - those that can be produced internally,  
- annotate genomes with KEGG Orthology terms (if not already done) to identify which KEGG Modules can be completed by neighboring taxa.  

In the following sections, we will provide a bottom-up description of how `microbetag` can be used as a modular Python library, mirroring the way the tool’s main pipeline operates.  

By the end of this tutorial, you should:  
- have a solid understanding of the methods applied,  
- be familiar with the `microbetag` interface for executing various tasks, and  
- explore a real-world case study that demonstrates the potential of these methods.  



------------------

One can use `microbetag` in two ways:

- **On-the-fly** – the software runs on a virtual machine at KU Leuven, and the user interacts with it through **Cytoscape** and the **MGG add-on**. This requires no installation on your part.  
- **Locally** – install the `microbetag` source code on your own machine. This removes the limitations of the on-the-fly version and also allows access to additional features not available remotely.  

> ⚠️ Note: Using `microbetag` locally can be more involved, as it requires installing several dependencies.

------

❗ This tutorial runs in a **GitHub Codespace**, giving you a hassle-free setup while allowing us to explore the full range of features that `microbetag` offers when installed locally—all from the cloud.

If you prefer to set up `microbetag` on your own computer, you can start with our [installation guide](https://microbetag.readthedocs.io/en/v1.0.4/installation.html).  
If you run into any issues, don’t hesitate to reach out on [Matrix](https://matrix.to/#/#microbetagcommunity:matrix.org).


The *on-the-fly* version will be covered through our [slides](https://docs.google.com/presentation/d/15WvhB9Vff3fWYVFUMFNaGt8xiR1J4lhcKyq7AeR3YVU/edit?usp=sharing). 

------

### Build the codespace

So, for now, here is what you need to do:


1. (Optional) Make a fork of our [`metabolic_toy_model` GitHub repository](https://github.com/hariszaf/metabolic_toy_model).  
   This gives you your own workspace where you can make changes, add new material, and experiment freely.

2. Make sure you are on the [`kul2025`](https://github.com/hariszaf/metabolic_toy_model/tree/kul2025) branch

3. Click on the *Code* button, and then from the *Codespaces* tab, click on the *Create codespace on kul2025*




<img src="figs/create_codespace.jpeg" width="20%">

> GitHub Codespace: 
>
> A **cloud-based, online** integrated development environment developed by GitHub. It allows users to create and manage development **environments directly within the browser** or through Visual Studio Code desktop.
> Development containers, or dev containers, are **Docker containers** that are specifically configured to provide a fully featured development environment. **Whenever you work in a codespace, you are using a dev container on a virtual machine**.

By now, a new tab should have popped up and your codespace should be under construction!

If you still see the following image on the bottom right of your screen, click on the `Building codespace...`.

<img src="figs/setting_up_codespace.png" width="20%">

This should open a terminal showing the *Details* of what’s happening during the build.  

The snapshot below shows an example taken at a random point in the process.  
Notice the two terminals on the right: **`Details`** and **`Configuration`**.  
You’ll only see the `Details` panel if you clicked on *Building codespace...* — but don’t worry if it’s not there. It’s optional and only provides extra information about what’s going on.


<img src="figs/codespace_building_time.png" width="60%">

otherwise, press `F1`, start typing `View Creation Log` and click on it. 

> ✍️ In case we have not mentioned this already, we are on VS Code. The `F1` button on a Codespace VS Code, pops up what is known as `Command Palette` on VS Code, from where you can do a great range of cool things! 

Last, let's have a closer look on what we see on the four corners of our screen! 

Can you locate the name of your codespace? Our GitHub repository and branch should be common for all of us, right?  The codespace ? 

### Let's have a walk around!  

Here is a the structure of this codespace:

In [ ]:
# You do not have to run this cell. You may execute it only after the next section `microbetag conda environment as your kernel` has been performed
!tree -aL 1 .

.
├── .devcontainer
├── .git
├── .gitignore
├── README.md
├── data
├── figs
├── licenses
├── microbetag
└── microbetag_tutorial.ipynb

7 directories, 3 files


> Those interested in how this codespace is being built, you may have a look at the `.devcontainer` 

You may observe that not all these folders are present in the branch you 
The `licenses/` folder ok, we just built it to get the Gurobi license. 
But what about the `microbetag` folder ?

This was actually retrieved during the building of the codespace and it's a clone of a certain branch called `codespace` from Haris' fork.

You may browse a bit to have a better view on what's there

In [ ]:
# Like in the previous cell; no need to run this cell. Execute it only if your have set your Kernel correct.
!tree -L 1 microbetag/

microbetag/
├── CONTRIBUTING.md
├── Dockerfile
├── Dockerfile.microbetag_base
├── LICENSE
├── MANIFEST.in
├── README.md
├── build
├── config_files
├── docs
├── environment.yml
├── ext_data
├── microbetag
├── microbetag.egg-info
├── requirements
├── setup.py
├── setup_environment.sh
├── test_data
└── tests

10 directories, 9 files


Under [`microbetag`](microbetag/microbetag/) is the source code of the python library. 

Under the [`config_files`](microbetag/config_files/), one can find a complete configuration file per microbetag version. 

For this tutorial though, we will mostly focus on the: 

- [`tests`](microbetag/tests/) : scripts to test `microbetag`'s features
- [`test_data`](microbetag/test_data) : input/output and configuration files for each of the tests under `tests/`



In [9]:
!tree -L 1 microbetag/tests/

microbetag/tests/
├── README.md
├── microbetag_prep_example.tsv
├── test_build_cx2.py
├── test_carve.py
├── test_faprotax.py
├── test_flashweave.py
├── test_kegg_annotation.py
├── test_manta.py
├── test_microbetag.py
├── test_modelseed.py
├── test_path_compl.py
├── test_phenotrex.py
├── test_prodigal.py
└── test_seed_compl.py

1 directory, 14 files


In [14]:
!tree -L 1 microbetag/test_data/test_build_cx2/

microbetag/test_data/test_build_cx2/
├── README.md
├── config_buildCX2.yml
├── input_files
└── output_files

3 directories, 2 files


> **Remember:**  
> 
> We will be using the test scripts under `tests/` **just to get to know** the resource! You do not have to follow them exactly.  
> 
> In fact, you can do much better! The purpose of these scripts is simply to **test some features**.  
> So feel free to explore and experiment **during or after the tutorial**!


### The `microbetag` `conda` environmnet as our notebook's kernel

When setting up this Codespace (via [`devcontainer.json`](.devcontainer/devcontainer.json) and [`setup_codespace.sh`](.devcontainer/setup_codespace.sh)) three separate conda environments were created.

However, `conda` environments don’t automatically register as Jupyter kernels; Jupyter only sees kernels that have been explicitly added with `ipykernel`.

To this end, before starting, you need to open a terminal (if not already opened on the bottom of your screen) and run the following commands **from there** -- **not** from within the notebook. 

🛑 Let us first check if the codespace is actually completed!

Do you see the `Outcome` in the last line ? 🎉

<img src="figs/codespace_build_complete.png" width="40%">

So, let us now make sure that Jupyter can "find" our environment:

`conda activate microbetag`

`conda install -y ipykernel`

`python -m ipykernel install --user --name=microbetag --display-name "Python (microbetag)"`

However, as we will see later on, during the building of this codespace, another `conda` environment was also built, to run the genome-based node annotation step. 
So, let us allow to Jupyter to access that too. 

`python -m ipykernel install --user --name=mtg-phenotrex --display-name "Python (phenotrex)"`

Once this is complete, you will have to refresh this tab, by pressing `F1` (or click the three lines on the top left and then `View > Command Palette..`) and then type `Reload Window`.

<img src="figs/reload_window.png" width="40%">

🤗 Assuming you are back, all you have to do now is to activate the proper Kernel on the notebook. To do so, try to run the following cell (either clicking the *play* button on the left of the cell once you click on it) or by `super + Enter`.

In [3]:
print("🦠 hello bac friend 🧫")

🦠 hello bac friend 🧫


Jupyter will ask you which kernel to use on top of your screen, click on `Python Environments...`

<img src="figs/select_kernel.png" width="40%">

and then, instead of the *recomended* one, click on the `microbetag (Python 3.10.0)`

<img src="figs/new_conda_env_among_kernels.png" width="40%">

Great! Now on top right of your window it should say something like *microbetag (Python 3.10)*! 

So, let us load some basic libraries that will be used for trivial tasks around the notebook. 

In [4]:
import pandas as pd

### Get a Gurobi license

In case you wish to reconstruct a GEM, either with ModelSEEDpy or CarveMe, or gap-fill one using DNNGIOR, in all these cases you need to have first get yourself a Gurobi license. 

Gurobi is **commercial** software (sorry for this) 
yet, there is a free license for academic purposes. 

If you wish to run this notebook locally, you need to make sure you get one first! You may [follow our instructions](https://github.com/hariszaf/metabolic_toy_model/blob/duth/prep_env.ipynb) for how to do this from a previous course. 




In [ ]:
# Create directory for the license
os.makedirs("licenses", exist_ok=True)

In [6]:
# Function to make sure you can use Gurobi on Colab
def create_gurobi_license(WLSACCESSID, WLSSECRET, LICENSEID):

    license_content = (
        "# Gurobi WLS license file\n"
        "# Your credentials are private and should not be shared or copied to public repositories.\n"
        "# Visit https://license.gurobi.com/manager/doc/overview for more information.\n"
        f"WLSACCESSID={WLSACCESSID}\n"
        f"WLSSECRET={WLSSECRET}\n"
        f"LICENSEID={LICENSEID}"
    )

    with open("licenses/gurobi.lic", "w") as f:
        f.write(license_content)
    print("License file created at licenses/gurobi.lic")

    # Set an environment variable inside Python 
    # -- Attention! This will hold only within this notebook and for the programs this notebook fires. 
    # You may set this globally on your terminal
    os.environ['GRB_LICENSE_FILE'] = 'licenses/gurobi.lic'

> 🔴 The following credentials will be valid just until the end of the summer school (by the end of Sept 2025 will be expired). 
> 
> Thus, we strongly suggest you either get a Gurobi Web License Service (WLS) academic license of your own and replace the following credentials, or you set everything locally. 
> In both cases, you can use [our instructions for how to get a Gurobi license](https://github.com/hariszaf/metabolic_toy_model/blob/duth/prep_env.ipynb).

In [7]:
WLSACCESSID = "815a008a-be80-4776-b3e3-22dafedf32ae"
WLSSECRET   = "f47c686a-ac3f-4483-b159-9e025dc71ff9"
LICENSEID   = 964844

In [9]:
create_gurobi_license(WLSACCESSID, WLSSECRET, LICENSEID)

License file created at licenses/gurobi.lic


In [10]:
import gurobipy as gbp

model = gbp.Model("test")
print("Gurobi is working!", "\U0001F600")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 964844
Academic license 964844 - for non-commercial use only - registered to ha___@kuleuven.be
Gurobi is working! 😀


### Install the `microbetag` library

We now have everything set! Yet, to use `microbetag` as a Python library, we can install it, by simply running:

In [ ]:
!cd microbetag && pip install .

You can check that this worked by checking whether `microbetag` can be invoked:

In [ ]:
!microbetag -h

> `microbetag` was built as an annotator tool, yet as you will discover during this tutorial, it implements a range of different tasks, by invoking a series of state-of-the-art tools
>
> We will try to partially cover those steps, so you can have a better understanding of all the required steps, but also how you can do this on your own, exploiting `microbetag` as a suite. 


----------

Go back to the slides to have a fair introduction in the basic concepts to be used. 📚 🤓

----------

## Let's build a co-occurrence network

Most of the times, one would probably use `microbetag` as a black box: 

- here is my abundance tabe -- and my genomes if available
- give me an annotated network 


In this tutorial though, we will follow a different approach. 

We will be using `microbetag` in a modular way, so we both learn:

- **the technical part:** *how to use* `microbetag` as a resource/platform for doing a range of tasks
- **the science part:** the *background* of each task and what we *insight* gain from it, but also its *challenges/limitations*

So, since `microbetag` was initially built as a co-occurrence network annotating tool, let's first build such a network! 

To this end, we will perform the [`test_flashweave.py`](microbetag/tests/test_flashweave.py) test. 

Let's have a look at this script! 👨🏻‍💻

It's quite clear that this script has a *"control room"*, the [`config_test_fw.yml`](microbetag/test_data/test_flashweave/config_test_fw.yml) file. 

So, let us have a look at that too! 👨🏻‍💻

From there, you may notice the parameter for the input abundance table, `abundance_table_file`, and the **metadata file**, `metadata_file`.

**The values of these arguments are paths to the corresponding files to be used as input**

Also, there is a list of FlashWeave related parameters, under the `flashweave_args` key, to better tune FlashWeave according to your dataset's features. 




So, now that we know what it is going on, let us fire our first network inference!

> For now on, you may either run the tests scripts through the notebook, or on the integrated VS code terminal. To switch from one to the other, you can type `ctrl+j`.

In [11]:
# This is necessary only (!) in case you wish to run the script from the notebook -- it makes sure that Jupyter notebook knows the path of julia 
os.environ["JULIA_BINDIR"] = "/home/vscode/.microbetag/julia-1.9.4/bin"
os.environ["PATH"]         = "/home/vscode/.microbetag/julia-1.9.4/bin:" + os.environ["PATH"]

In [ ]:
!python microbetag/tests/test_flashweave.py

Let us know check what's the output look like!

Keep in mind, the test script was supposed to build **two networks**; one using metadata as input and one without.

So, let's check the: 

- [`network_output.edgelist`](microbetag/test_data/test_flashweave/output_files/network_output.edgelist), and the
- [`network_metadata.edgelist`](microbetag/test_data/test_flashweave/output_files/network_metadata.edgelist)

under the `microbetag/test_data/test_flashweave/output_files` folder.

> By clicking on them, you will open these files on VS Code, within your codespace. You can edit them as you wish, and save them too; like any file on your codespace.
>
> If you have made a fork, and fired the codespace from that, then you can **commit** your changes, and **push** them on your fork.
>
> Last, you can **right** click on the files and then `Download` them, so you can access them locally! 🚀

<img src="figs/download_file.png" width="20%">

Let's have a look now at a network file. 

First, it would be a bit better to have column names, right? So, go ahead and give something like *nodeA*, *nodeB* and *weight*. 

It can be whatever, but **be careful** and use the same separator as in the following lines, i.e. tab -- an actual tab and not several spaces in a row! 😅

Then, go ahead and have a look at the last two rows of this file, can you think of anything funny with:

```
ASV0017	ASV0099	0.8098902702331543
ASV0099	ASV0017	0.9350579380989076
```

To make sure of that, let's see how our network actually looks like on Cytoscape! 

So fire your Cytoscape and then click 

`File > Import > Network from file...` or simply press `ctrl+l` (on Linux -- should be something similar on MacOS and Windows).

So we were right and you see something like this:

<img src="figs/double_edge.png" width="20%">

This is because we never checked the actual data right ? 
So, let's do that now, let's have a look at the [`testAbundMeta.tsv`](microbetag/test_data/test_flashweave/input_files/testAbundMeta.tsv) file.

Do you see anything funny? 🤓

In [13]:
abd_table = pd.read_csv("microbetag/test_data/test_flashweave/input_files/testAbundMeta.tsv")
abd_table.head()

,id,P-001,P-003,P-004,P-005,P-006,P-007,P-008,P-009,P-010,P-011,P-012,P-013,P-014,P-015,P-016,P-017,Taxonomy
0,ASV0003,1522,0,850,1238,0,0,0,1405,0,0,0,0,0,1465,0,0,Bacteria;Actinobacteriota;Actinobacteria;Bifid...
1,ASV0004,1396,1455,1379,861,1201,302,1495,15,719,405,415,299,1000,959,642,1642,Bacteria;Firmicutes;Bacilli;Lactobacillales;St...
2,ASV0005,1485,1632,2005,2229,1339,1979,1768,1375,1471,1034,1360,920,1225,1043,1115,1874,Bacteria;Actinobacteriota;Coriobacteriia;Corio...
3,ASV0012,531,454,525,488,403,619,372,263,290,191,256,207,320,225,220,393,Bacteria;Firmicutes;Clostridia;Oscillospirales...
4,srss::ASV0014,1587,1079,935,862,1341,1072,1081,364,593,465,313,376,761,709,640,1145,Bacteria;Firmicutes;Negativicutes;Veillonellal...


In [14]:
abd_table.tail()

,id,P-001,P-003,P-004,P-005,P-006,P-007,P-008,P-009,P-010,P-011,P-012,P-013,P-014,P-015,P-016,P-017,Taxonomy
73,ASV0957,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...
74,ASV1009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bacteria;Firmicutes;Negativicutes;Veillonellal...
75,ASV1071,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bacteria;unknown;unknown;unknown;unknown;unknown;
76,ASV1076,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bacteria;Actinobacteriota;Coriobacteriia;Corio...
77,ASV1079,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bacteria;Actinobacteriota;Coriobacteriia;Corio...


Yeap! A lot of zeroes — just like Prof. Faust discussed during her talk.  

So, if you look closely, you can see at least two odd things: 

- an association between ASV0655 that has 0s across all samples, and ASV0119 that does have some non-zero values, and
- a double edge between ASV0017 and ASV1079, again the latter having only zero values across all samples.

So, even we cannot say a lot about what is the double edge for the time being, what we observe here are **artifacts** and should **not** be there. 
Thus, this is our first **best practice**:

$\rightarrow$ **always remove from you input abundance table, all zero-only rows**

This is very easy to happen if you *crop* a dataset.
Even if a row had non-zero values in the original dataset, after cropping it might contain only zeroes.  


#### 📝 Task 1

So, now, let us do this as a first exercise! 

Let us make a copy of the initial abundance data file, remove the rows that include only zero values, and edit the the `microbetag` configuration file accordingly! 

You can also create a new output folder (under `microbetag/test_data/test_flashweave/`) so you save your new networks there, **or rename** the old ones, or let them be **replaced** by the new ones. 

Likewise, instead of editing the initial configuration file ([`config_test_fw.yml`](microbetag/test_data/test_flashweave/config_test_fw.yml)) you can have a copy of it and edit the copy. 
In that case, remember to **also edit** the test script, to use your new config file.

Once you are ready, you may build your new network just like before! 

<details>
<summary> 👉 Solution  </summary>

*You can do better on your own than this supposed hint!*

- Make a **copy** of initial abundance data:
```
cd /workspaces/metabolic_toy_model/
cd microbetag/test_data/test_flashweave/input_files/
cp testAbundMeta.tsv testAbundMetaNonZero.tsv
cd /workspaces/metabolic_toy_model/
```
- click [`testAbundMetaNonZero.tsv`](microbetag/test_data/test_flashweave/input_files/testAbundMetaNonZero.tsv)
- **delete** lines 29-45 and save file (ctrl+s)
- From your terminal (ctrl+j) **rename (backup) original networks**:

```
cd microbetag/test_data/test_flashweave/output_files/
cp network_metadata.edgelist dd_network_metadata.edgelist
cp network_output.edgelist dd_network_output.edgelist
cd /workspaces/metabolic_toy_model/
```

- click [`config_test_fw.yml`](microbetag/test_data/test_flashweave/config_test_fw.yml) and replace `testAbundMeta.tsv` with `testAbundMetaNonZero.tsv` on the 
  `abundance_table_file: file_path:`

- Run test script:

```
python microbetag/tests/test_flashweave.py
```


</details>


> Do you still have a double edge on your network? Any guesses what does it stand for ? 

<details>
<summary>💭 A plausible interpretation -- yet, who knows?  </summary>

    The conditional independence tests are not perfectly symmetric.

    So the test of ASV0017 | ASV0397 isn’t guaranteed to give exactly the same statistic as ASV0397 | ASV0017. 

    FlashWeave outputs both, but conceptually they map to the same edge.

</details>


## Node annotation

### Literature based

FAPROTAX consists of two parts:  

1. A **database** containing phenotypic and ecological functions, along with manual annotations from the literature that link species to specific processes.  
2. A **parser script** that takes any taxonomic abundance file and, based on the assigned taxonomies, maps the corresponding traits from the database.  


So, let's have a quick look at the first, [`FAPROTAX.txt`](microbetag/microbetag/mtg_maps_models/FAPROTAX_1.2.10/FAPROTAX.txt):

```
methanotrophy	elements:C,H; main_element:C; electron_donor:C; electron_acceptor:variable; aerobic:variable; exclusively_prokaryotic:yes; light_dependent:no
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
*Archaea*ANME-2D*									# DOI:10.1038/nature12375
*Methanoperedens*nitroreducens*						# DOI:10.1038/nature12375
*Methylococcaceae*									# DOI:10.1007/0-387-30745-1_15
*Methylocystaceae*									# DOI:10.1007/0-387-30745-1_15

```


In [ ]:
!ls microbetag/test_data/test_faprotax/input_files

In [ ]:
import pandas as pd
df  = pd.read_csv("microbetag/test_data/test_faprotax/input_files/thirty_Samples.tsv", sep="\t")
df.head()

Contrary to the FlashWeave case, we now have **no configuration file** among the input files.  

So… is a configuration file always required?  
**No — it is not.**  

The configuration file is provided mainly to make things easier when the number of arguments is large. For simple cases like this, you can just pass the required arguments directly in a script, as shown in the [`test_faprotax.py`](microbetag/tests/test_faprotax.py) example.  

Let's take a look at the script!


Did you notice the `Config` class? This is a **different** configuration class and has nothing to do with the:

In [ ]:
from microbetag.config import Config

we could name it whatever and would not change a thing. 

In [ ]:
!python microbetag/tests/test_faprotax.py

Let us see our findings! Frome the *Explorer* on your left, click on `microbetag > test_data > test_faprotax > output_files > faprotax`

or go there from your terminal. 

The outputs are a folder (`sub_tables`) and the `functional_otu_table.tsv` file. Let's start from one from the sub-tables!
I will open [`aerobic_ammonia_oxidation.txt`](microbetag/test_data/test_faprotax/output_files/faprotax/sub_tables/aerobic_ammonia_oxidation.txt) and you can open anyone random!

So, what we can see here are the **bins** that, based on their assigned **taxonomy**, were predicted to be able to oxidize ammonia **aerobically** — 
a process in which ammonia is converted to nitrite by **ammonia-oxidizing microorganisms (AOM)**. We also see their abundances in per sample.


In [17]:
from pathlib import Path
fapro_outdir = Path("microbetag/test_data/test_faprotax/output_files/faprotax")
sub_table    = pd.read_csv(fapro_outdir / "sub_tables/aerobic_ammonia_oxidation.txt", sep="\t")
sub_table.head()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,#
record,seqId,sample1,sample2,sample4,sample5,sample6,sample7,sample8,sample9,sample10,sample11,sample12,sample13,sample14,sample15,sample16,sample17,sample18,sample19,sample20,sample21,sample22,sample23,sample24,sample25,sample26,sample27,sample28,sample29,sample30
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Nitrosomonadaceae;g__Nitrosospira;s__Nitrosospira sp001899235,bin_176,93,74,4,29,44,57,31,5,19,5,91,70,24,70,84,27,75,67,64,33,4,66,24,3,85,84,31,37,68
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Nitrosomonadaceae;g__Nitrosospira;s__,bin_199,55,83,92,23,73,86,65,54,48,71,3,58,69,60,20,17,96,9,12,55,58,99,68,18,54,85,35,27,29


In [ ]:
fapro_out_table = pd.read_csv(fapro_outdir/ "functional_otu_table.tsv", sep="\t")
fapro_out_table.head()

However, the last one is not of interest to us.  
What we actually care about are the **sub-tables** that let us identify whether any of the taxa in our abundance table carry one or more of the **80+ traits** included in FAPROTAX.

#### 📝 Task 2 

Get the abundance table we used in the [*build a co-occurrence network*](#lets-build-a-co-occurrence-network) section and run FAPROTAX using this as input. 

<details>
<summary> 👉 Solution</summary>

All you have to do is to make the `abundance_table` variable on the `test_faprotax.py` script point at the correct abundance table file.

Also to make sure you provide the right column name as your taxonomy column, and last, that you set the abundance table delimiter correctly.

```
abundance_table = root_dir / "test_data/test_flashweave/input_files/testAbundMeta.tsv"
taxonomy_col = "Taxonomy"
delimiter = "\t"
```

You may rename the original output folder to anything else if you wish to keep it as a backup. 

You can now fire the test, the same way we did above: 

```python microbetag/tests/test_faprotax.py```

</details>

### Genome based

`phenotrex` is a bit tricky and requires a very specific set of versions for Python and some key libraries, such as `numpy`.  

Because of this, **a different `conda` environment** will be used, and for that, **we need to change the kernel**. 

The required environment was already built during the building of this codespace, and we added it among those that Jupyter can access earlier on this notebook. 

So, all you have to do now, is to press F1 and start typing *Select Notebook Kernel*, then, if not already there, click on the *Select Another Kernel..* and then *Jupyter kernel...*.
You should now be able to select *Python (phenotrex)*!

<img src="figs/pheno_kernel.png" width="40%">

> ⚠️ Switching kernels
>
> Switching kernels leads to **loss of all variables currently in memory**. That means that after you have switched your kernel, you will have to run any command loading for example some library and those keeping a variable you are interested in.



So, `phenotrex` operates on **presence/absence patterns of [eggNOG](http://eggnog5.embl.de/#/app/home) cluster IDs** in the passed genome. 

groups of orthologous genes (OGs) that have been clustered together based on evolutionary relationships.

a set of genes from different organisms that are likely derived from a common ancestor and share similar function.

👉 An eggNOG cluster ID is an identifier for a family of genes across organisms that are orthologous and predicted to perform the same or very similar function.


If a DNA FASTA file is passed to `phenotrex`, **Prodigal** is first used to find protein sequences - this step is skipped if a protein FASTA file is passed instead. 

To then find eggNOG cluster IDs from protein sequences, deepnog is used. Input files to feature creation may thus be DNA or protein multi-FASTA files, which may optionally be gzipped.

In [ ]:
!phenotrex -h

So let us have a quick view on the corresponding test script, [`test_phenotrex.py`](microbetag/tests/test_phenotrex.py).

In [ ]:
!python microbetag/tests/test_phenotrex.py

So now, we can see what's the output of our trait prediction! 

In [ ]:
!ls microbetag/test_data/test_phenotrex/output_files/

And let's have a look at each of those traits:

In [ ]:
!cat microbetag/test_data/test_phenotrex/output_files/aerobe.prediction.tsv

#### 📝 Task 3 

Do you see any funny file among your output files?

<details>
<summary> 👉 Solution</summary>

Have a look at the `microbetag/test_data/test_phenotrex/output_files/train.genotype` file!
</details>

Based on the challenge we just faced with `conda`, do you have any ideas about what is gonna be the next *big* change on our codebase?

<details>
<summary> 👉 Idea </summary>

Have a look at [nextflow](https://www.nextflow.io/)! 

</details>

## Pathway complementarity

In [ ]:
To get pathway complementarities, you first need to 

In [ ]:
!python microbetag/tests/test_kegg_annotation.py

In [ ]:
!python microbetag/tests/test_path_compl.py

Now, let's see what we did!! 

We got two output files:

- [`alternatives.json`](microbetag/test_data/test_path_compl/output_files/alternatives.json)
- [`complementarities.json`](microbetag/test_data/test_path_compl/output_files/complementarities.json)

In the first one, you may see all the different ways to enumerate modules that your species can go for 

In [57]:
import json 

with open("microbetag/test_data/test_path_compl/output_files/alternatives.json") as f:
    alternatives = json.load(f)

print(alternatives.keys())
print(alternatives["bin_101"].keys())
print(alternatives["bin_101"]["md:M00018"])

dict_keys(['bin_101', 'bin_151', 'bin_19', 'bin_38', 'bin_41', 'bin_45', 'bin_48'])
dict_keys(['md:M00001', 'md:M00002', 'md:M00003', 'md:M00004', 'md:M00006', 'md:M00008', 'md:M00009', 'md:M00010', 'md:M00011', 'md:M00012', 'md:M00013', 'md:M00014', 'md:M00015', 'md:M00016', 'md:M00017', 'md:M00018', 'md:M00019', 'md:M00020', 'md:M00021', 'md:M00023', 'md:M00024', 'md:M00025', 'md:M00026', 'md:M00027', 'md:M00028', 'md:M00029', 'md:M00030', 'md:M00031', 'md:M00032', 'md:M00033', 'md:M00034', 'md:M00035', 'md:M00036', 'md:M00037', 'md:M00038', 'md:M00039', 'md:M00040', 'md:M00042', 'md:M00043', 'md:M00044', 'md:M00046', 'md:M00047', 'md:M00048', 'md:M00050', 'md:M00051', 'md:M00053', 'md:M00055', 'md:M00056', 'md:M00057', 'md:M00058', 'md:M00059', 'md:M00060', 'md:M00061', 'md:M00063', 'md:M00064', 'md:M00065', 'md:M00066', 'md:M00067', 'md:M00068', 'md:M00069', 'md:M00070', 'md:M00071', 'md:M00072', 'md:M00073', 'md:M00074', 'md:M00075', 'md:M00076', 'md:M00077', 'md:M00078', 'md:M000

So what we can see here, is that for the [threoning biosynthesis](https://www.kegg.jp/module/M00018) module, M00018, there are **12 alternative ways** for a species to have it complete:


In [50]:
len(alternatives["bin_101"]["md:M00018"].keys())

12

And that for this bin, `bin_101`, in order to fill each of them, here what would be necessary:

In [ ]:
for alternative, required_kos in alternatives["bin_101"]["md:M00018"].items():
    print(alternative , "--", required_kos)

Based on this information, we can now check the `complementarities.json`:

In [ ]:
with open("microbetag/test_data/test_path_compl/output_files/complementarities.json") as f:
    complementarities = json.load(f)

In [58]:
complementarities["bin_101"].keys()
complementarities["bin_101"]["bin_38"][28]

['md:M00018',
 ['K00872', 'K12524', 'K00133'],
 ['K12524', 'K00133', 'K12524', 'K00872', 'K01733'],
 'https://www.kegg.jp/kegg-bin/show_pathway?map00260/K01733%09%23EAD1DC/K12524%09%2300A898/K00133%09%2300A898/K12524%09%2300A898/K00872%09%2300A898/']

## A real-world example with a subgingival plaque dataset

## Our data 

For this tutorial, we will use two different test cases

| Use case | Description | Purpose |
|----------|-------------|---------|
| **Synthetic community (4-species)** | A certain species is believed to have a positive effect on several others | Use `microbetag` locally with **our own genomes** to explore the concept of **complementarity** in detail |
| **Natural communities (hundreds of taxa)** | Co-occurrence network derived from communities across multiple samples | Use `microbetag` annotations with **network clustering** and **enrichment analysis** to **generate new hypotheses** on community drivers |


### Synthetic community

The following list of **obligate anaerobes**:

- *Anaerococcus vaginalis* (Av)
- *Finegoldia magna* (Fm)
- *Peptoniphilus asaccharolyticus* (Pa)

have been found to have a positive association with *Pseudomonas aeruginosa* (Pseud), a **facultative anaerobe**.

For all those four species, the online version of `microbetag` already has at least a GTDB representative genome, however, we will download their corresponding genomes so we can run an example of how one would use `microbetag` with their own, custom genomes.

To this end, let's have a look at the celebrated [GTDB](https://gtdb.ecogenomic.org/).


You may see, that there are several genomes from most of those genera, and in many cases of the exact species -- since we have no further information on what strains have this association we have to *guess* 🤷🏾  

| Strain | GTDB entry | NCBI entry | 
|:------:|:----------:|:----------:|
| Pseudomonas aeruginosa |  [GCF_001457615.1](https://gtdb.ecogenomic.org/genome?gid=GCF_001457615.1) | [GCA_001457615.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_001457615.1) |
| Finegoldia magna_H |  [GCF_000010185.1](https://gtdb.ecogenomic.org/genome?gid=GCF_000010185.1) | [GCA_000010185.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_000010185.1) |
| Anaerococcus vaginalis |  [GCF_000311745.1](https://gtdb.ecogenomic.org/genome?gid=GCF_000311745.1) | [GCA_000311745.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_000311745.1) |
| Peptoniphilus asaccharolyticus |  [GCF_900176115.1](https://gtdb.ecogenomic.org/genome?gid=GCF_900176115.1) | [GCA_900176115.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_900176115.1) |


One can get genomes, using their [NCBI Rest API](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/api/rest-api/), 

<!-- curl -X GET "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip' 

curl -L -o genome.zip "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip' -->

```
curl -L -o genomes.zip "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1%2CGCF_000010185.1%2CGCF_000311745.1%2CGCF_900176115.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip'
```



from the **Bacterial and Viral Bioinformatics Resource Center (BV-BRC)** [platform](https://www.bv-brc.org/).


In [ ]:
genomes = {
    "Pseud": "GCF_001457615.1",
    "Fm"   : "GCF_000010185.1",
    "Av"   : "GCF_000311745.1",
    "Pa"   : "GCF_900176115.1"
}

In [ ]:


https://microbetag.readthedocs.io/en/v1.0.4/tutorials_otf/large_data.html



## Seed complementarity

For a thorough basic intro, you may have a look on another [branch](https://github.com/hariszaf/metabolic_toy_model/tree/duth) of this repo, 
where you may check how to deal with a model as a Python object using the [`cobra` library](https://cobrapy.readthedocs.io/), as weel as several constraint-based methods.

<details>
<summary>Toggle hint! 😉</summary>

dasd
</details>
